In [ ]:
workspace_id = "2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a"
bronze_lakehouse_id = "33b93b7e-0899-43c6-81a8-654bc740aa32"

In [1]:
from pyspark.sql import functions as F

# --- CONFIGURATION & PARAMETERS ---
BRONZE_BASE_PATH = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/dbo"

# 1. Read raw Delta tables from Lakehouse_Bronze
df_raw_locations = spark.read.format("delta").load(f"{BRONZE_BASE_PATH}/openaq_locations_bronze")
df_raw_measurements = spark.read.format("delta").load(f"{BRONZE_BASE_PATH}/openaq_measurements_bronze")

# 2. Flatten nested JSON structures for station & sensor metadata (Dimension source)
df_exploded = df_raw_locations.withColumn("sensor", F.explode_outer("sensors"))

df_sensors_silver = (
    df_exploded
    .select(
        F.col("id").cast("long").alias("location_id"),
        F.trim(F.col("name")).alias("location_name"),
        F.col("locality"),
        F.col("timezone"),
        F.col("isMobile").cast("boolean").alias("is_mobile"),
        F.col("isMonitor").cast("boolean").alias("is_monitor"),
        F.col("coordinates.latitude").cast("double").alias("latitude"),
        F.col("coordinates.longitude").cast("double").alias("longitude"),
        F.col("country.id").cast("long").alias("country_id"),
        F.col("country.code").alias("country_code"),
        F.col("country.name").alias("country_name"),
        F.col("provider.name").alias("provider_name"),
        F.col("owner.name").alias("owner_name"),
        F.col("sensor.id").cast("long").alias("sensor_id"),
        F.col("sensor.name").alias("sensor_name"),
        F.col("sensor.parameter.name").alias("parameter_name"),
        F.col("sensor.parameter.displayName").alias("parameter_display_name"),
        F.col("sensor.parameter.units").alias("parameter_units")
    )
    .filter(
        F.col("location_id").isNotNull() &
        F.col("latitude").isNotNull() &
        F.col("longitude").isNotNull()
    )
    .dropDuplicates(["location_id", "sensor_id"])
)

(
    df_sensors_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("openaq_silver")
)
print(f"Saved openaq_silver table. Record count: {df_sensors_silver.count()}")

# 3. Clean, normalize, and aggregate daily NYC air quality measurements (Fact source)
df_measurements_cleaned = (
    df_raw_measurements
    .withColumn("measurement_date", F.to_date(F.col("measurement_date"), "yyyy-MM-dd"))
    .withColumn("date_key", F.date_format(F.col("measurement_date"), "yyyyMMdd").cast("int"))
    .withColumn("parameter_name", F.lower(F.trim(F.col("parameter_name"))))
    .filter(
        F.col("measurement_date").isNotNull() &
        F.col("avg_value").isNotNull() &
        (F.col("avg_value") >= 0) &
        (F.col("avg_value") < 1000)
    )
    .dropDuplicates(["sensor_id", "measurement_date", "parameter_name"])
)

df_measurements_silver = (
    df_measurements_cleaned
    .groupBy("date_key", "measurement_date", "parameter_name", "parameter_units")
    .agg(
        F.round(F.avg("avg_value"), 3).alias("daily_avg_value"),
        F.round(F.min("min_value"), 3).alias("daily_min_value"),
        F.round(F.max("max_value"), 3).alias("daily_max_value"),
        F.countDistinct("sensor_id").cast("int").alias("active_sensors_count")
    )
    .orderBy("measurement_date", "parameter_name")
)

(
    df_measurements_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("openaq_measurements_silver")
)

print(f"Saved openaq_measurements_silver table. Record count: {df_measurements_silver.count()}")
display(df_measurements_silver.limit(10))

StatementMeta(, 330f19ee-16e8-46d8-a6bf-6d8d9d1bcc85, 3, Finished, Available, Finished, False)

Saved openaq_silver table. Record count: 288
Saved openaq_measurements_silver table. Record count: 1463


SynapseWidget(Synapse.DataFrame, 08c540a3-fbbe-4f22-ade2-7a241fb85de5)